In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/small_bench/checkpoints/pre_cell_22.pickle

In [4]:
%%RecordEvent
%%time
### cell 22 ###
from sklearn.feature_extraction.text import CountVectorizer

def get_n_grams(n_grams, top_n=10):
    df_words = pd.DataFrame()
    for dt in tqdm(train['discourse_type'].unique()):
        # filter by discourse type
        df = train[train['discourse_type'] == dt]
        texts = df['discourse_text'].tolist()

        # build n-gram counts
        vec = CountVectorizer(
            lowercase=True,
            stop_words='english',
            ngram_range=(n_grams, n_grams)
        ).fit(texts)
        bag_of_words = vec.transform(texts)
        sum_words = bag_of_words.sum(axis=0)
        words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]

        # create a dataframe of (word, count), sort and take top_n
        cvec_df = pd.DataFrame.from_records(
            words_freq,
            columns=['words', 'counts']
        ).sort_values(by='counts', ascending=False)
        cvec_df.insert(0, 'Discourse_type', dt)
        cvec_df = cvec_df.iloc[:top_n, :]

        # concatenate results, preserving original indices
        df_words = pd.concat([df_words, cvec_df])

    return df_words

# compute bigrams
bigrams = get_n_grams(n_grams=2, top_n=10)
bigrams.head()

  0%|          | 0/7 [00:00<?, ?it/s]

CPU times: user 149 ms, sys: 28.2 ms, total: 177 ms
Wall time: 175 ms


,Discourse_type,words,counts
55,Lead,cell phones,9
88,Lead,texting driving,4
57,Lead,operating vehicle,3
62,Lead,cell phone,3
251,Lead,distracted driving,3


In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/cell_19_small/checkpoints/post_cell_22_try_8.pickle